# Практика: Структурированный вывод с Groq API

В этом ноутбуке мы научимся получать от LLM не просто текст, а **структурированные данные** в формате JSON, которые можно сразу использовать в коде.

**Финальное задание:** построить мини-пайплайн для автоматического разбора резюме — программа будет извлекать из произвольного текста структурированный профиль кандидата.

## 0. Получение API-ключа Groq

**Пошаговая инструкция:**

1. Перейдите на **[console.groq.com/keys](https://console.groq.com/keys)** и зарегистрируйтесь
2. В правом меню выберите раздел **"API Keys"**
3. Нажмите **"Create API Key"**, введите название (например, `groq-practice`) и нажмите **"Create"**
4. Скопируйте ключ — он начинается с `gsk-...`

> **Важно:** ключ показывается только один раз. Сохраните его сразу.
>
> Новые аккаунты получают бесплатные токены для тестирования.

Создайте файл `.env` в той же папке, что и этот ноутбук:

```
GROQ_API_KEY=gsk-ВАШ_КЛЮЧ_ЗДЕСЬ
```

## 1. Установка и настройка клиента

In [ ]:
!pip install groq python-dotenv

import os
import json
import time
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv('GROQ_API_KEY')
client = Groq(api_key=api_key)

print(f'Ключ Groq загружен: {api_key[:5] if api_key else "Не найден"}...')

## 2. Проблема: LLM возвращает неструктурированный текст

Попросим модель извлечь данные из текста обычным запросом — и посмотрим, почему такой вывод неудобен для программы.

In [ ]:
resume_text = """
Привет! Меня зовут Анна Соколова, мне 28 лет. Последние 4 года работаю
бэкенд-разработчиком в финтех-компании. Основной стек: Python, FastAPI,
PostgreSQL, Redis. Есть опыт с Docker и Kubernetes. Закончила МГТУ им. Баумана
по специальности «Прикладная математика». Ищу позицию Senior Python Developer
с зарплатой от 300 000 рублей. Готова к удалённой работе или гибриду.
"""

# Обычный запрос без требования структуры
response = client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[
        {'role': 'user', 'content': f'Извлеки основную информацию из резюме:\n{resume_text}'}
    ]
)

raw_output = response.choices[0].message.content
print('Вывод модели:')
print(raw_output)
print()

# Попытка использовать вывод в коде — неудача
print('Попытка получить зарплату из ответа:')
try:
    data = json.loads(raw_output)
    print(data['salary'])
except Exception as e:
    print(f'Ошибка: {e}')
    print('Неструктурированный текст нельзя напрямую использовать в коде!')

## 3. Решение: JSON-режим

Передадим параметр `response_format={'type': 'json_object'}` — и модель вернёт валидный JSON.

> **Правило:** слово `JSON` должно присутствовать в системном промпте — иначе API вернёт ошибку.

In [ ]:
system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
Схема ответа:
{
  "name": string,
  "age": number или null,
  "experience_years": number,
  "current_position": string,
  "skills": [string],
  "education": string или null,
  "target_position": string,
  "salary_from": number или null,
  "work_format": string
}
"""

response = client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': f'Обработай резюме:\n{resume_text}'}
    ],
    response_format={'type': 'json_object'}
)

candidate = json.loads(response.choices[0].message.content)

print('Структурированный профиль кандидата:')
print(json.dumps(candidate, ensure_ascii=False, indent=2))
print()

# Теперь данные легко использовать в коде
print(f"Имя: {candidate['name']}")
print(f"Опыт: {candidate['experience_years']} лет")
print(f"Навыки: {', '.join(candidate['skills'])}")
print(f"Ожидаемая зарплата: {candidate['salary_from']:,} руб.".replace(',', ' '))

## 4. Пайплайн: обработка нескольких резюме

Завернём логику в функцию и запустим её на нескольких резюме сразу.

In [ ]:
import groq as groq_lib

def parse_resume(text: str) -> dict | None:
    """Извлекает структурированные данные из текста резюме."""
    system_prompt = """
    Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
    Схема: {"name": string, "experience_years": number, "skills": [string],
    "target_position": string, "salary_from": number или null, "work_format": string}
    """
    try:
        response = client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f'Обработай резюме:\n{text}'}
            ],
            response_format={'type': 'json_object'}
        )
        return json.loads(response.choices[0].message.content)
    except groq_lib.RateLimitError as e:
        print(f'Лимит запросов. Ждём 5 сек... ({e})')
        time.sleep(5)
        return None
    except Exception as e:
        print(f'Ошибка: {e}')
        return None


resumes = [
    """
    Иван Петров, 32 года. 7 лет в разработке игр на Unity и C#.
    Делал мобильные игры для iOS/Android, есть опыт с Blender.
    Ищу Lead Game Developer, готов к офису в Москве. Зарплата от 250 000.
    """,
    """
    Мария Иванова. Data Scientist с опытом 3 года.
    Стек: Python, pandas, scikit-learn, PyTorch, SQL.
    Работала в e-commerce над рекомендательными системами.
    Хочу ML Engineer, только удалёнка, зарплата не принципиальна.
    """,
    """
    Дмитрий Козлов, 25 лет, junior frontend. Знаю React, TypeScript, CSS.
    Полгода коммерческого опыта в веб-студии. Ищу работу в продуктовой компании.
    Рассматриваю от 120 000, гибрид или удалёнка.
    """,
    """
    Екатерина Смирнова, 29 лет. DevOps-инженер, 5 лет опыта.
    Умею: Kubernetes, Docker, Terraform, AWS, CI/CD (GitLab, GitHub Actions).
    Работала в банковском секторе, есть опыт с highload-системами.
    Ищу Senior DevOps, удалёнка обязательно. Зарплата от 300 000.
    """,
    """
    Алексей Новиков, 35 лет. Fullstack-разработчик, 10 лет опыта.
    Стек: Python (Django, FastAPI), Vue.js, PostgreSQL, Redis, Docker.
    Последние 3 года — тимлид команды из 6 человек в SaaS-компании.
    Рассматриваю роль Engineering Manager или Staff Engineer. От 400 000, гибрид.
    """,
    """
    Юлия Белова. QA Engineer, 4 года в тестировании.
    Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests).
    Опыт в agile-командах, знакома с нагрузочным тестированием (Locust).
    Ищу Middle/Senior QA, гибрид, зарплата от 180 000.
    """,
    """
    Сергей Орлов, 27 лет. Android-разработчик, 3 года опыта.
    Kotlin, Jetpack Compose, Room, Retrofit, Coroutines. Публиковал 2 приложения в Google Play.
    Немного знаю iOS (Swift). Ищу Middle Android Developer в продуктовую компанию.
    Только удалёнка. Зарплата от 210 000.
    """
]

results = []
for i, resume in enumerate(resumes, 1):
    print(f'Обрабатываю резюме {i}/{len(resumes)}...')
    parsed = parse_resume(resume)
    if parsed:
        results.append(parsed)

print(f'\nОбработано: {len(results)} из {len(resumes)} резюме')
print()
for r in results:
    salary = f"{r['salary_from']:,}".replace(',', ' ') + ' руб.' if r.get('salary_from') else 'не указана'
    print(f"• {r['name']} | {r['target_position']} | опыт {r['experience_years']} л. | зарплата: {salary}")


## 5. Фильтрация и анализ собранных данных

Структурированные данные позволяют легко фильтровать кандидатов без дополнительных запросов к модели.

Ниже показан один пример фильтрации — изучите его, затем **выполните самостоятельные задания** в следующей ячейке.


In [ ]:
# Пример: кандидаты с опытом 3+ лет и зарплатой до 300 000
filtered = [
    r for r in results
    if r['experience_years'] >= 3
    and (r['salary_from'] is None or r['salary_from'] <= 300_000)
]

print(f'Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): {len(filtered)}')
for r in filtered:
    print(f"  → {r['name']}: {', '.join(r['skills'][:3])}...")

print()

# Самый опытный кандидат
if results:
    top = max(results, key=lambda r: r['experience_years'])
    print(f"Самый опытный: {top['name']} ({top['experience_years']} лет опыта)")

# Все упомянутые навыки
all_skills = [skill for r in results for skill in r['skills']]
unique_skills = sorted(set(all_skills))
print(f'\nВсе навыки в пуле: {len(unique_skills)}')
print(', '.join(unique_skills))


### Задание: фильтрация кандидатов

Используя список `results`, выполните следующие задачи в ячейке ниже:

1. **Удалёнщики-синьоры** — найдите кандидатов, у которых `work_format` содержит слово `удалёнка` и `experience_years >= 4`.
2. **Бюджетный найм** — выведите всех кандидатов с зарплатными ожиданиями не выше 230 000 руб. (или без указанной зарплаты).
3. **Python-разработчики** — отберите кандидатов, у которых в `skills` есть `Python`.
4. **Топ-3 по опыту** — выведите трёх самых опытных кандидатов в порядке убывания стажа.
5. **Статистика зарплат** — посчитайте среднюю, минимальную и максимальную ожидаемую зарплату среди тех, кто её указал.


In [ ]:
# Задание 1: удалёнщики с опытом 4+ лет
# TODO: напишите list comprehension или цикл


# Задание 2: зарплатные ожидания <= 230 000 (или не указаны)
# TODO:


# Задание 3: кандидаты с Python в навыках
# Подсказка: используйте any() или проверку 'Python' in r['skills']
# TODO:


# Задание 4: топ-3 по опыту
# Подсказка: sorted(..., key=..., reverse=True)[:3]
# TODO:


# Задание 5: статистика зарплат
# Подсказка: сначала соберите список salary_from тех, у кого он не None
# TODO:


## 6. Сравнение моделей

Groq предоставляет доступ к нескольким моделям с разными характеристиками. Сравним их по **скорости** и **качеству** извлечения структурированных данных.

| Модель | Размер | Типичная скорость |
|---|---|---|
| `llama-3.1-8b-instant` | 8B | ~1000–1500 токенов/с |
| `llama-3.3-70b-versatile` | 70B | ~250–400 токенов/с |
| `gemma2-9b-it` | 9B | ~500–900 токенов/с |
| `mixtral-8x7b-32768` | 8×7B MoE | ~400–700 токенов/с |

> 💡 Актуальный список моделей всегда доступен на [console.groq.com/docs/models](https://console.groq.com/docs/models)


In [ ]:
import time

# Резюме для теста
test_resume = """
Андрей Захаров, 31 год. Backend-разработчик, 6 лет опыта.
Основной стек: Go, Python, gRPC, Kafka, PostgreSQL, Kubernetes.
Строил высоконагруженные сервисы в финтех и логистике.
Ищу Senior/Lead Backend Engineer. Удалёнка, зарплата от 380 000.
"""

system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
Схема: {"name": string, "experience_years": number, "skills": [string],
"target_position": string, "salary_from": number или null, "work_format": string}
"""

models_to_compare = [
    'llama-3.1-8b-instant',
    'llama-3.3-70b-versatile',
    'gemma2-9b-it',
    'mixtral-8x7b-32768',
]

comparison_results = []

for model_name in models_to_compare:
    try:
        start = time.time()
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f'Обработай резюме:\n{test_resume}'}
            ],
            response_format={'type': 'json_object'}
        )
        elapsed = time.time() - start

        parsed = json.loads(response.choices[0].message.content)
        usage = response.usage

        comparison_results.append({
            'model': model_name,
            'time_sec': round(elapsed, 2),
            'total_tokens': usage.total_tokens,
            'tokens_per_sec': round(usage.completion_tokens / elapsed),
            'skills_found': len(parsed.get('skills', [])),
            'parsed': parsed,
        })
        print(f'OK {model_name}: {elapsed:.2f} с, {usage.total_tokens} токенов')
        time.sleep(1)  # пауза между запросами
    except Exception as e:
        print(f'ОШИБКА {model_name}: {e}')

print()
header = f"{'Модель':<35} {'Время':>8} {'Ток/с':>8} {'Навыков':>9}"
print(header)
print('-' * 65)
for r in comparison_results:
    row = f"{r['model']:<35} {r['time_sec']:>7.2f}с {r['tokens_per_sec']:>8} {r['skills_found']:>9}"
    print(row)


### Задание: анализ результатов сравнения

После выполнения ячейки выше ответьте на вопросы (в новой ячейке типа Markdown или в комментариях):

1. Какая модель оказалась **самой быстрой** по времени ответа? А по числу токенов в секунду?
2. Все ли модели извлекли одинаковое количество навыков? Есть ли расхождения?
3. Когда стоит предпочесть большую модель (70B) маленькой (8B)? Приведите пример задачи.
4. **Бонус:** добавьте в список ещё одну модель из документации Groq и запустите сравнение заново.


---
## Самостоятельное задание: парсер вакансий

Вы видели, как разбирать **резюме**. Теперь сделайте то же самое для **вакансий**.

### Задача
Реализуйте функцию `parse_vacancy(text: str) -> dict | None`, которая принимает текст вакансии и возвращает структурированный JSON со следующими полями:

```json
{
  "position": "название должности",
  "company": "название компании или null",
  "required_skills": ["список обязательных навыков"],
  "nice_to_have": ["список желательных навыков"],
  "experience_required": "например '3-5 лет' или null",
  "salary_to": "верхняя граница зарплаты или null",
  "work_format": "офис / удалёнка / гибрид"
}
```

### Требования
- Используйте `response_format={'type': 'json_object'}`
- Системный промпт должен содержать слово `JSON` и описание схемы
- Оберните вызов в `try/except`
- Проверьте функцию на всех пяти тестовых вакансиях ниже

### Бонус ⭐
Напишите функцию `match_candidates(vacancy: dict, candidates: list) -> list`, которая по уже разобранным данным подбирает подходящих кандидатов из списка `results` (совпадение хотя бы 2 навыков из `required_skills`).

In [ ]:
# Тестовые вакансии
vacancy_1 = """
Компания TechFlow ищет Senior Python Developer.
Обязательно: Python 5+ лет, FastAPI или Django, PostgreSQL, Docker.
Желательно: Kubernetes, опыт с микросервисами, Redis.
Условия: гибрид (Москва), зарплата до 400 000 рублей, ДМС.
"""

vacancy_2 = """
Ищем Middle Data Scientist в e-commerce стартап.
Требования: Python, pandas, scikit-learn, опыт построения ML-моделей от 2 лет.
Плюсом будет: PyTorch, опыт с рекомендательными системами, A/B-тесты.
Полная удалёнка, зарплата до 220 000. Команда 5 человек.
"""

vacancy_3 = """
Frontend Developer (React) — продуктовая компания в сфере EdTech.
Must have: React, TypeScript, REST API, опыт от 1 года.
Nice to have: Next.js, тестирование (Jest), Storybook.
Офис в СПб или удалёнка, вилка 150 000 – 200 000.
"""


def parse_vacancy(text: str) -> dict | None:
    """
    Извлекает структурированные данные из текста вакансии.

    Args:
        text: Текст вакансии в произвольном формате

    Returns:
        Словарь с полями position, company, required_skills,
        nice_to_have, experience_required, salary_to, work_format
        или None при ошибке
    """
    # TODO: Реализуйте функцию
    # Подсказки:
    # 1. Системный промпт: опишите роль и схему JSON
    # 2. Не забудьте слово JSON в системном промпте!
    # 3. Передайте text в user-сообщение
    # 4. Используйте response_format={'type': 'json_object'}
    # 5. Верните json.loads(response.choices[0].message.content)

    pass  # Удалите эту строку и напишите свой код


# Тестирование
for i, vacancy in enumerate([vacancy_1, vacancy_2, vacancy_3], 1):
    print(f'{'=' * 55}')
    print(f'Вакансия {i}')
    print('=' * 55)
    result = parse_vacancy(vacancy)
    if result:
        print(json.dumps(result, ensure_ascii=False, indent=2))
    print()

In [ ]:
# Бонус: подбор кандидатов под вакансию

def match_candidates(vacancy: dict, candidates: list) -> list:
    """
    Подбирает кандидатов из списка по совпадению навыков.

    Args:
        vacancy: Разобранная вакансия (результат parse_vacancy)
        candidates: Список разобранных резюме (результаты parse_resume)

    Returns:
        Список подходящих кандидатов с указанием совпавших навыков
    """
    # TODO: Реализуйте функцию
    # Подсказки:
    # 1. Возьмите required_skills из vacancy
    # 2. Для каждого кандидата найдите пересечение его skills с required_skills
    # 3. Верните кандидатов, у которых совпало >= 2 навыков
    # 4. В возвращаемый словарь добавьте поле matched_skills

    pass

---
## Чек-лист перед сдачей

- [ ] Все ячейки выполняются без ошибок (Kernel → Restart & Run All)
- [ ] Раздел 5: выполнены все 5 заданий по фильтрации
- [ ] Раздел 6: таблица сравнения моделей заполнена, письменные ответы добавлены
- [ ] Функция `parse_vacancy()` реализована и возвращает корректный JSON для всех 5 вакансий
- [ ] Все 7 полей схемы присутствуют в ответе (или `null`, если данных нет)
- [ ] Используется `response_format={'type': 'json_object'}`
- [ ] В коде есть обработка ошибок `try/except`
- [ ] API-ключ не хранится в коде — только через `.env`
- [ ] (Бонус) Функция `match_candidates()` реализована
